# Part 1 - Fine-Tuning YOLO26 for Construction-Site Safety (PPE Detection)

On a busy construction site, a safety officer simply **cannot watch every worker, every second**, to check that each one is wearing a hard hat and a high-visibility vest. But a camera plus a fine-tuned object detector **can** - flagging anyone missing protective gear instantly, around the clock, across every camera on site.

In this hands-on tutorial (**Part 1** of the series) we build exactly that, step by step.

## What you'll learn
- What **fine-tuning** is, and why a pretrained model isn't enough on its own
- How to pull a real **PPE-detection dataset** (already in YOLO format) into your notebook
- How to **fine-tune YOLO26** on it in a few minutes
- How to **evaluate** the model (mAP, confusion matrix, training curves)
- How to run it on **images and video**, with a live **"SAFETY VIOLATION"** alert

## Why fine-tune? (the whole point)
The pretrained YOLO26 model (trained on the COCO dataset) already detects a generic **`person`** - but it has **no idea whether that person is wearing a hard hat or a safety vest**. For safety compliance, *"is there a person?"* is the wrong question; the real one is *"is this person properly protected?"*

Checking that by hand, for dozens of workers across multiple site cameras, is impossible for a human to do reliably. A model can do it tirelessly and instantly. To give the model this new skill, we **teach it the safety-specific classes it has never seen** - `Hardhat`, `Safety Vest`, `NO-Hardhat`, and so on. Taking a capable pretrained model and teaching it a new, specialised task on a smaller dataset is exactly what **fine-tuning** means.


## 1. Setup
Install the libraries and run Ultralytics' environment check (it prints versions and shows the GPU if one is available).

In [ ]:
%pip install -q ultralytics roboflow gdown
import ultralytics
ultralytics.checks()

## 2. Configuration

In [ ]:
import os, sys

# --- Model + dataset ---
MODEL_WEIGHTS = "yolo26n.pt"          # nano = fast; try yolo26s.pt / yolo26m.pt for more accuracy
WORKSPACE = "roboflow-universe-projects"
PROJECT   = "construction-site-safety"
VERSION   = 27
FORMAT    = "yolov8"
EPOCHS    = 100
IMGSZ     = 640
RUN_NAME  = "ppe_finetune"

# --- Test assets (images + videos) ---
# Pulled from a public GitHub Release - no API key, no Drive login needed.
ASSETS_DIR = "assets"
ASSETS_URL = "https://github.com/spmallick/learnopencv/releases/download/v1.0/Assets.zip"

IN_COLAB = "google.colab" in sys.modules
print("Environment:", "Colab" if IN_COLAB else "local / Kaggle / other")
print("Model:", MODEL_WEIGHTS, "| Dataset:", PROJECT, "v" + str(VERSION))

## 3. Roboflow API key
The dataset lives on Roboflow, which needs a **free** API key. This helper finds it in whatever environment you run in - no Colab-only code.

In [ ]:
# Get a FREE key at roboflow.com → Settings → API Keys, then paste it here:
API_KEY = "0c4XQBsECsYHA7WKoJnW"

assert API_KEY != "PASTE_YOUR_ROBOFLOW_KEY", "Paste your Roboflow API key above."

## 4. Download the dataset
Pulls the **Construction Site Safety** dataset (Roboflow, CC BY 4.0) already in YOLO format - images, labels and a ready `data.yaml`, no conversion needed.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(FORMAT)
DATA_YAML = os.path.join(ds.location, "data.yaml")
print("Dataset ready at:", ds.location)

## 5. Download the test assets

In [ ]:
import glob, urllib.request, zipfile

# Pull the test images/videos from the public GitHub Release (once). No key, no login.
if not glob.glob(os.path.join(ASSETS_DIR, "**", "*"), recursive=True):
    urllib.request.urlretrieve(ASSETS_URL, "Assets.zip")
    zipfile.ZipFile("Assets.zip").extractall(ASSETS_DIR)

def asset(name):
    """Resolve a test file by NAME only - no hardcoded paths, handles any sub-folder."""
    hits = glob.glob(os.path.join(ASSETS_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {ASSETS_DIR}/ - check ASSETS_URL.")
    return hits[0]

print("Assets folder:", os.path.abspath(ASSETS_DIR))

## 6. Explore the dataset
A quick look before training: the classes, how many images per split, the class balance, and a few labelled samples.

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(open(DATA_YAML))
_names = cfg["names"]
NAMES = {int(k): v for k, v in _names.items()} if isinstance(_names, dict) else {i: n for i, n in enumerate(_names)}
print(f"Classes ({len(NAMES)}):", NAMES)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_images(d):
    p = Path(d)
    return [q for q in p.rglob("*") if q.suffix.lower() in IMG_EXT] if p.exists() else []
def split_dir(split):
    for c in [Path(ds.location)/split/"images", Path(ds.location)/split]:
        if c.exists(): return c
    return Path(ds.location)/split/"images"

TRAIN_IMAGES = list_images(split_dir("train"))
VAL_IMAGES   = list_images(split_dir("valid")) or list_images(split_dir("val")) or list_images(split_dir("test"))
print("Train images:", len(TRAIN_IMAGES), "| Val images:", len(VAL_IMAGES))

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def yolo_label_path(p):
    parts = list(Path(p).parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"; break
    return Path(*parts).with_suffix(".txt")

counts = Counter()
for img in TRAIN_IMAGES:
    lp = yolo_label_path(img)
    if lp.exists():
        for line in lp.read_text().splitlines():
            if line.strip():
                counts[int(float(line.split()[0]))] += 1
if counts:
    labels = [NAMES.get(c, str(c)) for c in sorted(counts)]
    vals   = [counts[c] for c in sorted(counts)]
    plt.figure(figsize=(10, 4)); plt.bar(labels, vals, color="#F4A300")
    plt.title("Training boxes per class"); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
import cv2, random

def draw_boxes(p):
    im = cv2.imread(str(p))
    if im is None: return None
    h, w = im.shape[:2]; lp = yolo_label_path(p)
    if lp.exists():
        for line in lp.read_text().splitlines():
            if not line.strip(): continue
            c, xc, yc, bw, bh = map(float, line.split()[:5])
            x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
            x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
            cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(im, str(NAMES.get(int(c), "")), (x1, max(12, y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1, cv2.LINE_AA)
    return im[:, :, ::-1]

samples = random.sample(TRAIN_IMAGES, min(6, len(TRAIN_IMAGES)))
plt.figure(figsize=(14, 8))
for i, p in enumerate(samples):
    rgb = draw_boxes(p)
    if rgb is None: continue
    plt.subplot(2, 3, i + 1); plt.imshow(rgb); plt.axis("off")
plt.suptitle("Sample training images with ground-truth PPE boxes"); plt.tight_layout(); plt.show()

## 7. First, watch the base model FAIL (this is the "why")
Before we train anything, let's prove the point. The stock YOLO26 knows COCO's 80 everyday classes - it will happily box a `person`, but it has **no class** for a hard hat or a vest. So for safety compliance it is useless out of the box. Confirming this is the whole motivation for fine-tuning.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

base = YOLO(MODEL_WEIGHTS)
coco = set(base.names.values())
missing = [n for n in NAMES.values() if n not in coco]
print(f"{len(missing)} of {len(NAMES)} PPE classes are NOT in COCO - the base model is blind to them:")
print(missing)

# Find an image where the BASE (COCO) model itself detects a PERSON,
# so "sees a person but misses the PPE" is guaranteed to show on screen.
pool = (VAL_IMAGES or TRAIN_IMAGES)
sample, res = None, None
for p in pool[:80]:                      # scan up to 80 images, stop at the first hit
    rr = base.predict(str(p), conf=0.35, verbose=False)[0]
    labels = [base.names[int(c)] for c in rr.boxes.cls.tolist()] if (rr.boxes is not None and len(rr.boxes)) else []
    if "person" in labels:
        sample, res = p, rr
        break
if sample is None:                       # fallback if none found
    sample = pool[0]; res = base.predict(str(sample), conf=0.25, verbose=False)[0]

plt.figure(figsize=(9, 6)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis("off")
plt.title("Pretrained YOLO26 (COCO): boxes the worker as 'person' - but misses all PPE"); plt.show()
print("Image used:", sample)

## 8. Fine-tune YOLO26
We start from the pretrained `yolo26n.pt` (so we keep everything it already learned about shapes and edges) and teach it the safety classes. On a GPU this takes only a few minutes. All outputs land under `runs_yolo26/`.

In [ ]:
model = YOLO(MODEL_WEIGHTS)
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20,
    project="runs_yolo26", name=RUN_NAME, exist_ok=True,
)
SAVE_DIR = Path(model.trainer.save_dir)
print("Training complete ->", SAVE_DIR)

## 9. Evaluate
mAP scores plus the plots Ultralytics saves automatically - training curves, the confusion matrix, and the precision-recall curve. (Watch the `NO-*` classes: asserting that gear is *absent* is the hardest part.)

In [ ]:
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f} | mAP50: {metrics.box.map50:.4f}")

from IPython.display import Image, display
for pattern in ["results.png", "confusion_matrix.png", "*PR_curve.png"]:
    for f in sorted(SAVE_DIR.glob(pattern)):
        print(f.name); display(Image(filename=str(f)))

## 10. Test it on images
Run the fine-tuned model on our test images (pulled by name from `assets/`): a compliant crew, a no-hard-hat violation, and a busy site with vests + machinery.

In [ ]:
import cv2
best = YOLO(SAVE_DIR / "weights" / "best.pt")
names = [
    "yolo26_Finetuning_image_PPE_hardhat_workers.jpg",
    "yolo26_Finetuning_image_PPE_no_hardhat.jpg",
    "yolo26_Finetuning_image_PPE_vest_and_machinery.jpg",
]

H = 640
imgs = []
for n in names:
    a = cv2.imread(asset(n))
    s = H / a.shape[0]
    imgs.append(cv2.resize(a, (int(a.shape[1] * s), H)))

preds = best.predict(imgs, conf=0.20, verbose=False)
anns = [r.plot(line_width=2, font_size=16) for r in preds]

ws = [a.shape[1] for a in anns]
fig_w = 14
fig, axes = plt.subplots(1, 3, figsize=(fig_w, fig_w * H / sum(ws)),
                         gridspec_kw={"width_ratios": ws})
for ax, a in zip(axes, anns):
    ax.imshow(a[:, :, ::-1]); ax.axis("off")
plt.suptitle("Fine-tuned YOLO26 - PPE detection on test images")
plt.subplots_adjust(wspace=0.02, left=0, right=1, top=0.92, bottom=0)
plt.show()

## 11. Real-time video + a "SAFETY VIOLATION" alert
Finally, run it on a site video. Any `NO-*` detection (e.g. `NO-Hardhat`) flashes a red banner - the actual alerting behaviour a real deployment would use.

In [ ]:
PPE_VIDEO = asset("yolo26_Finetuning_video_PPE_man_phone.mp4")   # or _man_phone / _site_timelapse
best = YOLO(SAVE_DIR / "weights" / "best.pt")
NAMES = best.names
VIOLATION = [n for n in NAMES.values() if str(n).upper().startswith("NO-")]
print("Violation classes:", VIOLATION)

OUT = "ppe_annotated.mp4"
cap = cv2.VideoCapture(PPE_VIDEO); fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, writer = 1280, None
while True:
    ok, frame = cap.read()
    if not ok: break
    frame = cv2.resize(frame, (W, int(W * frame.shape[0] / frame.shape[1])))
    r = best.predict(frame, conf=0.25, verbose=False)[0]
    ann = r.plot()
    labs = [NAMES[int(c)] for c in r.boxes.cls.tolist()] if (r.boxes is not None and len(r.boxes)) else []
    if any(l in VIOLATION for l in labs):
        cv2.rectangle(ann, (0, 0), (ann.shape[1], 55), (0, 0, 255), -1)
        cv2.putText(ann, "SAFETY VIOLATION", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
    if writer is None:
        h2, w2 = ann.shape[:2]
        writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w2, h2))
    writer.write(ann)
cap.release()
if writer: writer.release()
print("Saved:", OUT)

In [ ]:
# Play the result inline. Re-encode to H.264 if ffmpeg is present (Colab/Kaggle have it); otherwise keep the raw mp4.
import shutil
play = OUT
if shutil.which("ffmpeg"):
    import subprocess
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUT,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", "ppe_play.mp4"])
    play = "ppe_play.mp4"

from IPython.display import HTML
from base64 import b64encode
HTML('<video width=720 controls><source src="data:video/mp4;base64,' +
     b64encode(open(play, "rb").read()).decode() + '" type="video/mp4"></video>')

## The real test: a full construction site

Clean, single-subject shots are a nice warm-up, but a real site is messy. There are several workers at once, some with hard hats and some without, plus safety cones, machinery, and vehicles all crowded into one frame. So let's push the model harder and run it on a complete construction scene packed with objects.

This is where it really counts. Can the model still pick out every person, flag the ones missing a hard hat, and catch the cones, machinery, and vehicles, all at the same time? Here is how it holds up on a busy, real-world site.

In [ ]:
PPE_VIDEO = asset("yolo26_Finetuning_video_PPE_site_timelapse.mp4")   # or _man_phone / _site_timelapse
best = YOLO(SAVE_DIR / "weights" / "best.pt")
NAMES = best.names
VIOLATION = [n for n in NAMES.values() if str(n).upper().startswith("NO-")]
print("Violation classes:", VIOLATION)

OUT = "ppe_annotated.mp4"
cap = cv2.VideoCapture(PPE_VIDEO); fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, writer = 1280, None
while True:
    ok, frame = cap.read()
    if not ok: break
    frame = cv2.resize(frame, (W, int(W * frame.shape[0] / frame.shape[1])))
    r = best.predict(frame, conf=0.25, verbose=False)[0]
    ann = r.plot()
    labs = [NAMES[int(c)] for c in r.boxes.cls.tolist()] if (r.boxes is not None and len(r.boxes)) else []
    if any(l in VIOLATION for l in labs):
        cv2.rectangle(ann, (0, 0), (ann.shape[1], 55), (0, 0, 255), -1)
        cv2.putText(ann, "SAFETY VIOLATION", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
    if writer is None:
        h2, w2 = ann.shape[:2]
        writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w2, h2))
    writer.write(ann)
cap.release()
if writer: writer.release()
print("Saved:", OUT)

In [ ]:
# Play the result inline. Re-encode to H.264 if ffmpeg is present (Colab/Kaggle have it); otherwise keep the raw mp4.
import shutil
play = OUT
if shutil.which("ffmpeg"):
    import subprocess
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUT,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", "ppe_play.mp4"])
    play = "ppe_play.mp4"

from IPython.display import HTML
from base64 import b64encode
HTML('<video width=720 controls><source src="data:video/mp4;base64,' +
     b64encode(open(play, "rb").read()).decode() + '" type="video/mp4"></video>')

## 12. Export for deployment
YOLO26's NMS-free head makes edge export clean (ONNX is the most portable).

In [ ]:
best.export(format="onnx")   # also: "engine" (TensorRT), "coreml", "tflite", "openvino"

## Recap & what's next

You just:
1. Saw **why** the base model can't do PPE compliance (it has no safety classes)
2. **Fine-tuned YOLO26** on a real construction-safety dataset
3. Evaluated it and ran it on **images and video** with a live violation alert

That's a complete, deployable safety-monitoring prototype - the kind of system a site manager would actually want.

**Part 2** applies the *same* fine-tuning workflow to a very different problem - real-time **hand-gesture / sign-language** recognition - so you can see how transferable the recipe is.

**Going further:**
- Add **tracking** (`model.track(...)`) so each worker is followed across frames and a violation is logged per person.
- Try a bigger model (`yolo26s`/`yolo26m`) or more epochs.
- Collect a few images from *your* site to sharpen accuracy on your specific cameras.
